In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import mlflow
import mlflow_utils as mltuti
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

# Environment setup
mlflow.set_tracking_uri("file:///tmp/mlflow_data")
mlflow.set_experiment("Ames_Housing_Analysis") 

# Load and split data
df = pd.read_csv('artifacts/train_clean.csv')
cols_to_drop = [col for col in df.columns if 'price' in col.lower() or col == 'Id']
X = df.drop(columns=cols_to_drop)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

# Log transform targets for better regression performance
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Dataset ready: {X_train.shape[0]} training samples.")

Dataset ready: 1164 training samples.


/usr/local/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## Baseline Model: Linear Regression
Before we try to optimize our predictions with tuning, we start with a **Linear Regression** baseline. This gives us a starting point for our **RMSE** (error) and **R²** (accuracy) metrics. We will log this run to MLflow so we can compare it against our later experiments.

In [2]:
with mltuti.start_mlflow_run(experiment_name="Ames_Housing_Analysis", 
                             run_name="Linear_Regression_Baseline"):
    
    # Initialize and train the linear regression model
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train_log)
    
    # Predict on the test set
    y_pred_lr = lr_model.predict(X_test)
    
    # Log the results
    mltuti.log_regression_metrics(y_test_log, y_pred_lr)
    
    # Log the model type
    mlflow.log_param("model_type", "LinearRegression")
    
    print("Run completed for linear regression.")

Run completed for linear regression.


## Comparing Models: Ridge vs. Linear Regression
Standard Linear Regression can sometimes "overfit" if there are too many features. **Ridge Regression** solves this by adding a penalty (Alpha) to the model. We will use MLflow to track how different Alpha values affect our **RMSE** and **R²**.

In [3]:
# Define range of alphas to test
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

for a in alphas:
    with mltuti.start_mlflow_run(experiment_name="Ames_Housing_Analysis", 
                                 run_name=f"Ridge_Alpha_{a}"):
        
        model = Ridge(alpha=a)
        model.fit(X_train, y_train_log)
        y_pred = model.predict(X_test)
        
        # Log metrics using our custom utility helper!
        mltuti.log_regression_metrics(y_test_log, y_pred)
        
        # Log specific parameters for this run
        mlflow.log_param("alpha", a)
        mlflow.sklearn.log_model(model, "housing_ridge_model")

        print(f"Run completed for Alpha: {a}")

2026/05/09 00:03:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 00:03:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run completed for Alpha: 0.01


2026/05/09 00:03:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 00:03:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/09 00:03:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 00:03:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

Run completed for Alpha: 0.1


2026/05/09 00:03:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 00:03:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run completed for Alpha: 1.0


2026/05/09 00:03:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run completed for Alpha: 10.0


2026/05/09 00:03:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run completed for Alpha: 100.0


## Conclusion
By reviewing the MLflow dashboard, we can see how **RMSE** and **R²** change as **Alpha** increases. This tracking allows us to pick the most "reproducible" model for production.